In [1]:
!pip install -U wandb
!pip install -q scikit-learn librosa

import os
import torch
import pandas as pd
import librosa
import numpy as np
import random
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.models as models
from tqdm import tqdm
from sklearn.metrics import f1_score
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("key")
wandb.login(key=wandb_key)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 72.1 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.0
    Uninstalling wandb-0.25.0:
      Successfully uninstalled wandb-0.25.0


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2007692 (24f2007692-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
csv_path = '/kaggle/input/notebooks/kartikbajaj1322/data-generation-for-resnet/resnet_metadata.csv'
base_dir = '/kaggle/input/notebooks/kartikbajaj1322/data-generation-for-resnet/resnet_wav_data'

df = pd.read_csv(csv_path)
labels = sorted(df['genre'].unique())
label2id = {l: i for i, l in enumerate(labels)}
df['label'] = df['genre'].map(label2id)

class ResNetAudioDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(base_dir, row['filename'])
        
        y, _ = librosa.load(path, sr=16000)
        
        target_len = 114176
        if len(y) > target_len:
            y = y[:target_len]
        else:
            y = np.pad(y, (0, target_len - len(y)))
            
        mel = librosa.feature.melspectrogram(y=y, sr=16000, n_fft=1024, hop_length=512, n_mels=224)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        scaled = np.clip((mel_db + 80.0) / 80.0, 0, 1)
        
        t = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0)
        
        target_w = 224
        if t.shape[-1] > target_w:
            t = t[:, :, :target_w]
        elif t.shape[-1] < target_w:
            t = F.pad(t, (0, target_w - t.shape[-1]))
            
        img = t.repeat(3, 1, 1)
        img = self.norm(img)
        
        return img, int(row['label'])


In [3]:
val_fold = 0

train_df = df[df['fold'] != val_fold]
val_df = df[df['fold'] == val_fold]

train_loader = DataLoader(ResNetAudioDataset(train_df), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(ResNetAudioDataset(val_df), batch_size=16, shuffle=False, num_workers=2)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


Train batches: 500, Val batches: 125


In [4]:
wandb.init(project="24f2007692-t12026", name="Resnet_50-Fold_2")

model = models.resnet50(weights='IMAGENET1K_V1')
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_ftrs, 10)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

epochs = 30
best_f1 = 0.0

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss = total_loss + loss.item()
        
    scheduler.step()
    
    model.eval()
    preds_list = []
    labels_list = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            
            _, preds = torch.max(outputs, 1)
            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
            
    val_f1 = f1_score(labels_list, preds_list, average='macro')
    avg_loss = total_loss / len(train_loader)
    
    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f} Val F1: {val_f1:.4f}")
    wandb.log({'epoch': epoch + 1, 'train_loss': avg_loss, 'val_f1': val_f1})
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'resnet50_best_fold.pth')
        
wandb.finish()
print("Fold Training Complete!")


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260329_095046-ypdwadzp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Resnet_50-Fold_2
wandb: ⭐️ View project at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026
wandb: 🚀 View run at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026/runs/ypdwadzp


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 210MB/s]
Epoch 1: 100%|██████████| 500/500 [03:48<00:00,  2.19it/s]


Epoch 1 Loss: 1.2801 Val F1: 0.8064


Epoch 2: 100%|██████████| 500/500 [02:49<00:00,  2.95it/s]


Epoch 2 Loss: 0.9025 Val F1: 0.8470


... (54 outputs left out)

Epoch 30: 100%|██████████| 500/500 [03:07<00:00,  2.67it/s]
wandb: updating run metadata


Epoch 30 Loss: 0.5206 Val F1: 0.9218


wandb: 
wandb: Run history:
wandb:      epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb: train_loss █▅▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:     val_f1 ▁▃▇▇▇█████████████████▇███████
wandb: 
wandb: Run summary:
wandb:      epoch 30
wandb: train_loss 0.52058
wandb:     val_f1 0.92181
wandb: 
wandb: 🚀 View run Resnet_50-Fold_2 at: https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026/runs/ypdwadzp
wandb: ⭐️ View project at: https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260329_095046-ypdwadzp/logs


Fold Training Complete!


In [ ]:
wandb.init(project="24f2007692-t12026", name="Resnet_50-Full_2")

full_loader = DataLoader(ResNetAudioDataset(df), batch_size=16, shuffle=True, num_workers=2)

final_model = models.resnet50(weights='IMAGENET1K_V1')
num_ftrs = final_model.fc.in_features
final_model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_ftrs, 10)
)
final_model = final_model.to(device)

optimizer = optim.AdamW(final_model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

for epoch in range(epochs):
    final_model.train()
    total_loss = 0.0
    
    for inputs, labels in tqdm(full_loader, desc=f"Full Epoch {epoch+1}"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss = total_loss + loss.item()
        
    scheduler.step()
    avg_loss = total_loss / len(full_loader)
    print(f"Full Epoch {epoch+1} Loss: {avg_loss:.4f}")
    wandb.log({'epoch': epoch + 1, 'train_loss': avg_loss})
    
torch.save(final_model.state_dict(), 'resnet50_final.pth')
wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260329_114323-jwqpfshn
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Resnet_50-Full_2
wandb: ⭐️ View project at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026
wandb: 🚀 View run at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026/runs/jwqpfshn
Full Epoch 1: 100%|██████████| 625/625 [03:42<00:00,  2.80it/s]


Full Epoch 1 Loss: 1.2401


Full Epoch 2: 100%|██████████| 625/625 [03:46<00:00,  2.77it/s]


Full Epoch 2 Loss: 0.8714


Full Epoch 3: 100%|██████████| 625/625 [03:50<00:00,  2.71it/s]


Full Epoch 3 Loss: 0.6773


... (51 outputs left out)

Full Epoch 29 Loss: 0.5159


Full Epoch 30: 100%|██████████| 625/625 [03:46<00:00,  2.76it/s]


Full Epoch 30 Loss: 0.5166


wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:      epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb: train_loss █▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:      epoch 30
wandb: train_loss 0.5166
wandb: 
wandb: 🚀 View run Resnet_50-Full_2 at: https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026/runs/jwqpfshn
wandb: ⭐️ View project at: https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260329_114323-jwqpfshn/logs
